In [1]:
!pip install -q langchain langchain_google_genai google_generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 8.6 MB/s eta 0:00:00


In [3]:
!pip install langchain-core langchain-community langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 2.6 MB/s eta 0:00:00


In [4]:
import os #for creating virtual environment
from langchain_google_genai import ChatGoogleGenerativeAI #for configuring our LLM
from langchain_core.prompts import ChatPromptTemplate #for building sequential workflows

In [5]:
#setting up an environment to connect with Google Gemini
os.environ["GOOGLE_API_KEY"]="API"

In [6]:
#initialize and configure the LLM
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain

In [9]:
#Chain 1 - Generating a Title from input topic from the user
title_prompt=PromptTemplate(
    input_variables=['topic'], #this topic will come from the user
    template="Create a catchy blog title about {topic}"
    )

title_chain=LLMChain(
    llm=llm,
    prompt=title_prompt,
    output_key="title" #from the 1st chain, we are expecting a title as output
)

/tmp/ipykernel_1429/981817131.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title_chain=LLMChain(


In [10]:
#Chain 2 - Generating an outline from the Title
outline_prompt=PromptTemplate(
    input_variables=['title'], #this has been generated as output from the previous chain
    template="Create a structured outline for a blog post titled {title}"
)

outline_chain=LLMChain(
    llm=llm,
    prompt=outline_prompt,
    output_key="outline"
)

In [12]:
#Chain 3 - Expand the outline into a Blog Draft
blog_prompt=PromptTemplate(
    input_variables=['outline'],
    template="Provide a detailed blog post based on the following outline in less than 300 words: {outline}"
)

blog_chain=LLMChain(
    llm=llm,
    prompt=blog_prompt,
    output_key="blog"
)

In [15]:
#Chain 4 - Summarize the Blog Draft
summary_prompt=PromptTemplate(
    input_variables=['blog'],
    template="Provide a concise summary of the following blog post: {blog}"
)

summary_chain=LLMChain(
    llm=llm,
    prompt=summary_prompt,
    output_key="summary"
)

In [16]:
overall_chain=SequentialChain(
    chains=[title_chain, outline_chain, blog_chain, summary_chain],
    input_variables=['topic'], #only one input from the user
    output_variables=['title', 'outline', 'blog', 'summary']
)

In [17]:
result=overall_chain({"topic":"AI driven Cybersecurity in Banking domain"})

/tmp/ipykernel_1429/4271148459.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result=overall_chain({"topic":"AI driven Cybersecurity in Banking domain"})


In [18]:
print(result['summary'])

This blog post emphasizes the critical importance of a compelling title to grab attention and boost engagement for complex topics like AI-driven cybersecurity in banking. A strong title also sets expectations, aids SEO, and builds brand authority.

The post provides a curated list of impactful blog titles, categorized by their angle:
*   **Intriguing & Forward-Looking:** Sparks curiosity about innovation and future trends.
*   **Benefit-Oriented & Reassuring:** Focuses on positive outcomes, security, and trust.
*   **Action-Oriented & Dynamic:** Conveys urgency, proactivity, and strength against threats.
*   **Short & Punchy:** Designed for immediate impact and memorability.

It concludes by advising authors to consider their audience, core message, tone, and platform when choosing a title to align perfectly with their content and strategic goals.


Activity - Telecom Customer Complaint Analysis

Chain 1 - Takes raw input customer complaint and does a complaint classification into category, severity and customer sentiment

Chain 2 - Takes classification category from chain 1 to create a root cause analysis

Chain 3 - Generate a recommendation or resolution plan

Chain 4 - Create a summary of the previously generated resolution plan